# Badge 3: account setup

Run once, before Lesson 1. The course lab is a Snowflake trial account where the learner has ACCOUNTADMIN, SYSADMIN and the warehouse COMPUTE_WH. This account is shared, so these cells create stand-ins:

| Course | Here |
|---|---|
| ACCOUNTADMIN | `BADGE3_DABW_ADMIN` |
| SYSADMIN | `BADGE3_DABW_ROLE` |
| COMPUTE_WH | `BADGE3_DABW_WH` |

Account-level steps the course does as ACCOUNTADMIN (resource monitor, DORA setup) run as the real ACCOUNTADMIN and are labelled. Object names (SMOOTHIES, FRUIT_OPTIONS, ORDERS, the two apps, UTIL_DB functions) are the course's.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE3_DABW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE3_DABW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE3_DABW_WH"      # stands in for the course's COMPUTE_WH
import ast, io
from snowflake.snowpark.context import get_active_session

CUSTOM = 'SMOOTHIES.PUBLIC."Custom Smoothie Order Form"'
PENDING = 'SMOOTHIES.PUBLIC."Pending Smoothie Orders"'


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result


def write_app(app, app_code):
    """SQL equivalent of editing the app in Snowsight's Streamlit editor: write streamlit_app.py into the app's live version."""
    ast.parse(app_code)  # the code must at least be valid Python
    path = f"snow://streamlit/{app}/versions/live/streamlit_app.py"
    get_active_session().file.put_stream(io.BytesIO(app_code.encode()), path, auto_compress=False, overwrite=True)
    print(f"wrote {len(app_code.splitlines())} lines to {path}")

## Check for name collisions
Course database names are account-wide.

In [ ]:
%%sql -r existing_course_objects
use role accountadmin;
show databases like 'SMOOTHIES';

## Roles and grants

In [ ]:
%%sql -r create_roles
use role accountadmin;
create role if not exists {{ADMIN_ROLE}} comment = 'Badge 3 stand-in for ACCOUNTADMIN';
create role if not exists {{SYSADMIN_ROLE}} comment = 'Badge 3 stand-in for SYSADMIN';
grant role {{SYSADMIN_ROLE}} to role {{ADMIN_ROLE}};
grant role {{ADMIN_ROLE}} to role sysadmin;
set badge_user = current_user();
grant role {{ADMIN_ROLE}} to user identifier($badge_user);
grant role {{SYSADMIN_ROLE}} to user identifier($badge_user);
-- SYSADMIN creates the SMOOTHIES database (Lesson 2)
grant create database on account to role {{SYSADMIN_ROLE}};
-- ACCOUNTADMIN can read SNOWFLAKE.ACCOUNT_USAGE (DABW005 reads query_history)
grant imported privileges on database snowflake to role {{ADMIN_ROLE}};

## Warehouse (stands in for COMPUTE_WH; the Lesson 2 checklist says SYSADMIN owns it)

In [ ]:
%%sql -r create_warehouse
use role accountadmin;
create warehouse if not exists {{WAREHOUSE}}
  warehouse_size = xsmall auto_suspend = 60 auto_resume = true initially_suspended = true
  comment = 'Badge 3 stand-in for COMPUTE_WH';
grant ownership on warehouse {{WAREHOUSE}} to role {{SYSADMIN_ROLE}} copy current grants;

## UTIL_DB (shared across badges; home of the DORA GRADER and, in Lesson 8, the course's functions)

In [ ]:
%%sql -r util_db_grants
use role accountadmin;
create database if not exists UTIL_DB comment = 'Shared utility database for badge course audits (DORA grader, course stages)';
grant ownership on database UTIL_DB to role sysadmin copy current grants;
grant ownership on schema UTIL_DB.PUBLIC to role sysadmin copy current grants;
grant usage on database UTIL_DB to role {{ADMIN_ROLE}};
grant usage on database UTIL_DB to role {{SYSADMIN_ROLE}};
grant usage, create stage, create workspace on schema UTIL_DB.PUBLIC to role {{ADMIN_ROLE}};
-- Lesson 8: the SYSADMIN stand-in creates its functions in UTIL_DB.PUBLIC
grant usage, create function on schema UTIL_DB.PUBLIC to role {{SYSADMIN_ROLE}};

## Course data file (the learner's Downloads folder)
Lesson 2 has the learner download `fruits_available_for_smoothies.txt`. The file is in `data/`; upload it once:

```
snow sql -q "use role BADGE3_DABW_ADMIN; put 'file://<path>/data/*' @UTIL_DB.PUBLIC.BADGE3_DOWNLOADS auto_compress=false overwrite=true"
```

In [ ]:
%%sql -r create_downloads_stage
use role {{ADMIN_ROLE}};
create stage if not exists UTIL_DB.PUBLIC.BADGE3_DOWNLOADS
  comment = 'Badge 3: stands in for the learner''s local Downloads folder'
  directory = (enable = true)
  encryption = (type = 'SNOWFLAKE_SSE');
grant read on stage UTIL_DB.PUBLIC.BADGE3_DOWNLOADS to role {{SYSADMIN_ROLE}};
alter stage UTIL_DB.PUBLIC.BADGE3_DOWNLOADS refresh;
list @UTIL_DB.PUBLIC.BADGE3_DOWNLOADS;

In [ ]:
expected = ['fruits_available_for_smoothies.txt']
found = sorted(n.split('/')[-1] for n in pdf(create_downloads_stage)['name'])
print('Missing course files:', sorted(set(expected) - set(found)) or 'none')

## Teardown (commented out)
Removes everything Badge 3 creates.

In [ ]:
%%sql -r teardown
-- use role accountadmin;
-- drop database if exists SMOOTHIES;
-- drop function if exists UTIL_DB.PUBLIC.SUM_MYSTERY_BAG_VARS(number, number, number);
-- drop function if exists UTIL_DB.PUBLIC.NEUTRALIZE_WHINING(text);
-- drop resource monitor if exists Four_Daily_Credits;
-- drop stage if exists UTIL_DB.PUBLIC.BADGE3_DOWNLOADS;
-- drop warehouse if exists {{WAREHOUSE}};
-- drop role if exists {{SYSADMIN_ROLE}};
-- drop role if exists {{ADMIN_ROLE}};
select 'teardown is commented out' as status;